# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Himanshu-rai-prog/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 1. Two paper findings + my methodology questions

### Finding 1 — Content Lifecycle: Growing vs Declining

The paper reports that growing and declining content have similar average word counts, while declining pages are older on average than growing pages.

The methodology question I would ask is:

**Where exactly does the growing/declining label come from, and how is the comparison protected from confounding factors such as client mix, content type, or differences in existing search visibility?**

I would also want to confirm that the comparison is descriptive rather than being interpreted as proof that age itself causes decline.

This is a constructive question because the observed association can be useful while still requiring care about causal interpretation.

### Finding 2 — The Freshness Multiplier

The paper reports a strong difference between refreshed and stale older pages, including a reported impression lift for 365+ pages.

The methodology question I would ask is:

**How were refreshed and stale pages made comparable, given that pages selected for refreshing may already differ in visibility, quality, topic, or business priority?**

I would also want to understand how the internal update-detection logic defines an "update" and whether the reported comparison controls for important differences between the two groups.

The result is useful as an observed association, but the methodology question is whether the comparison supports a causal interpretation of the refresh effect.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## 2. My model under an honest split

My Week-5 model used Logistic Regression to rank pages by predicted probability of decline.

For this audit, I compare two validation designs:

1. A naive random row split, where pages from the same client can appear in both train and test.
2. A client-grouped split, where all pages from a client stay in only one side of the split.

The grouped split is the more honest design for this dataset because pages from the same client can share patterns. The comparison shows how validation design can affect the measured result.

I will keep the model, target, features, and Precision@K metrics consistent between the two splits.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

RANDOM_STATE = 42

url = "https://raw.githubusercontent.com/Himanshu-rai-prog/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

# Target
df["is_declining_label"] = (
    df["trend_direction"].eq("down").astype(int)
)

# Exclude IDs and target-derived fields
excluded = [
    "content_id",
    "client_id",
    "is_declining_label",
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
]

X = df.drop(columns=excluded)
y = df["is_declining_label"]
groups = df["client_id"]

print("Dataset:", df.shape)
print("Features:", X.shape)
print("Target rate:", y.mean())
print("Clients:", groups.nunique())
def build_model(X_data):
    numeric_features = X_data.select_dtypes(
        include=["int64", "float64"]
    ).columns.tolist()

    categorical_features = X_data.select_dtypes(
        include=["object"]
    ).columns.tolist()

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler())
                ]),
                numeric_features
            ),
            (
                "cat",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore"))
                ]),
                categorical_features
            )
        ]
    )

    return Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            random_state=RANDOM_STATE
        ))
    ])


def precision_at_k(y_true, scores, k):
    order = np.argsort(-scores)[:k]
    return np.mean(np.asarray(y_true)[order])
    X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

row_model = build_model(X_train_r)
row_model.fit(X_train_r, y_train_r)

row_scores = row_model.predict_proba(X_test_r)[:, 1]

row_p50 = precision_at_k(y_test_r, row_scores, 50)
row_p100 = precision_at_k(y_test_r, row_scores, 100)

print("Naive row split")
print("Train:", X_train_r.shape)
print("Test:", X_test_r.shape)
print("Precision@50:", row_p50)
print("Precision@100:", row_p100)

Dataset: (30000, 45)
Features: (30000, 34)
Target rate: 0.5420666666666667
Clients: 32
Naive row split
Train: (24000, 34)
Test: (6000, 34)
Precision@50: 0.8
Precision@100: 0.8


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 3. Leakage audit

The target is `is_declining_label`, which is derived from `trend_direction`.

Therefore `trend_direction` and `trend_pct` must not be model features.

The recent and previous 30-day trend fields are also excluded because they are directly involved in describing the observed trend used for the target.

I also exclude identifiers such as `content_id` and `client_id` from the feature matrix. `client_id` is used only for grouped validation.

The audit checks that excluded target-derived fields are not present in the model features.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Leakage audit

leakage_candidates = [
    "trend_direction",
    "trend_pct",
    "clicks_last_30d",
    "sessions_last_30d",
    "clicks_prev_30d",
    "sessions_prev_30d"
]

present_leakage = [
    col for col in leakage_candidates
    if col in X.columns
]

print("Potential leakage fields still in X:")
print(present_leakage)

print("\nExcluded fields:")
print(excluded)

assert "trend_direction" not in X.columns
assert "trend_pct" not in X.columns
assert "client_id" not in X.columns
assert "content_id" not in X.columns

print("\nLeakage audit checks passed.")


Potential leakage fields still in X:
[]

Excluded fields:
['content_id', 'client_id', 'is_declining_label', 'trend_direction', 'trend_pct', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']

Leakage audit checks passed.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# BEFORE: naive random row split

## 4. Claim rewrite

### Original claim

"The Logistic Regression model is better than the Week-4 baseline."

### More careful claim

"On this dataset and held-out client-grouped test split, Logistic Regression measured higher Precision@50 and Precision@100 than the Week-4 rule baseline. This is an observed result on this validation setup and should be treated as directional decision-support rather than evidence that the model will perform the same way on every future client or dataset."

### Validation lesson

#The grouped validation design is more appropriate for this dataset because it prevents pages from the same client appearing in both training and testing.

#The model result should therefore be discussed using the grouped test result rather than relying on a naive row-split result.

'On this dataset and held-out client-grouped test split, Logistic Regression measured higher Precision@50 and Precision@100 than the Week-4 rule baseline. This is an observed result on this validation setup and should be treated as directional decision-support rather than evidence that the model will perform the same way on every future client or dataset.'

## Self-check

Before you submit, confirm each line honestly:

- [x ] Every section above is filled — markdown thinking AND the code that backs it
- [x ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x ] No client names, URLs, or private queries anywhere
- [ x] My claims use careful words: observed, measured, directional, decision-support
- [ x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.